# Kalman Filter Scenario Plots

Two scenarios using the Kalman filter implementation from `kalman_filter.ipynb`:
1. **Constant Velocity** — well-tuned baseline
2. **Circular Trajectory** — model mismatch, CV filter applied to a manoeuvring target

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 20,
})

In [ ]:
# Core Kalman filter functions — from kalman_filter.ipynb

def predict(x, P, F, Q):
    x_pred = F @ x
    P_pred = F @ P @ F.T + Q
    return x_pred, P_pred

def update(x_pred, P_pred, z, H, R):
    y = z - H @ x_pred
    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T @ np.linalg.inv(S)
    x_upd = x_pred + K @ y
    I = np.eye(P_pred.shape[0])
    P_upd = (I - K @ H) @ P_pred
    return x_upd, P_upd, y, S

def kalman_filter(true_states, x, P, F, Q, H, R):
    x = x.copy()
    P = P.copy()

    true_positions = []
    measurements = []
    estimates = []
    covariances = []
    predictions = []
    prediction_covariances = []
    nis_log = []
    S_trace_log = []
    S_model_trace_log = []

    for k in range(len(true_states)):
        true_state = true_states[k]
        true_pos = H @ true_state

        noise_std = np.sqrt(R[0, 0])
        z = true_pos + np.random.normal(0, noise_std, size=2)

        x_pred, P_pred = predict(x, P, F, Q)
        predictions.append(x_pred)
        prediction_covariances.append(P_pred)
        x, P, y, S = update(x_pred, P_pred, z, H, R)
        S_model_trace_log.append(np.trace(H @ P_pred @ H.T))
        S_trace_log.append(np.trace(S))

        y = np.atleast_2d(y).reshape(-1, 1)
        nis = (y.T @ np.linalg.solve(S, y)).item()
        nis_log.append(nis)

        true_positions.append(true_pos)
        measurements.append(z)
        estimates.append([x[0], x[2]])
        covariances.append(P)

    return (
        np.array(true_positions), np.array(measurements), np.array(estimates),
        np.array(covariances), np.array(predictions), np.array(prediction_covariances),
        np.array(nis_log), np.array(S_trace_log), np.array(S_model_trace_log)
    )

In [ ]:
# Trajectory generators — from kalman_filter.ipynb

def constant_velocity_true_position(num_steps, dt, v_x, v_y):
    true_states = []
    for k in range(num_steps):
        t = k * dt
        px = v_x * t
        py = v_y * t
        true_states.append(np.array([px, v_x, py, v_y]))
    return np.array(true_states)

def circular_true_position(num_steps, dt, radius, omega):
    """Uniform circular motion: px = R·cos(ω·t), py = R·sin(ω·t).
    The CV model cannot represent sustained turning, so this is a genuine
    structural limitation rather than a tuning problem."""
    true_states = []
    t = 0.0
    for _ in range(num_steps):
        t += dt
        px =  radius * np.cos(omega * t)
        vx = -radius * omega * np.sin(omega * t)
        py =  radius * np.sin(omega * t)
        vy =  radius * omega * np.cos(omega * t)
        true_states.append(np.array([px, vx, py, vy]))
    return np.array(true_states)

In [ ]:
# Baseline parameters

dt = 1.0
num_steps = 60

# Velocities
v_x, v_y = 2.0, 1.0

# Initial state (start exactly on truth for constant-velocity scenarios)
x0 = np.array([0.0, v_x, 0.0, v_y])

# Initial covariance — moderate uncertainty
P0 = np.diag([1.0, 1.0, 1.0, 1.0])

# Process model — constant velocity
F = np.array([[1, dt, 0,  0 ],
              [0,  1, 0,  0 ],
              [0,  0, 1, dt ],
              [0,  0, 0,  1 ]])

# Process noise — sigma_acc = 0.1 m/s^2
sigma_acc = 0.1
q = sigma_acc ** 2
Q = np.array([[dt**4/4, dt**3/2,       0,       0],
              [dt**3/2,   dt**2,       0,       0],
              [      0,       0, dt**4/4, dt**3/2],
              [      0,       0, dt**3/2,   dt**2]]) * q

# Measurement matrix — observe position only
H = np.array([[1, 0, 0, 0],
              [0, 0, 1, 0]])

# Measurement noise — 1 m std
R = np.diag([1.0, 1.0])

## Scenario 1: Constant Velocity

The true trajectory follows constant velocity (`v_x=2.0, v_y=1.0` m/s). The filter model, process noise, and measurement noise are all well-matched. This is the baseline — showing the KF working as intended.

In [ ]:
np.random.seed(42)
true_states_cv = constant_velocity_true_position(num_steps, dt, v_x, v_y)

true_pos_cv, meas_cv, est_cv, cov_cv, pred_cv, pred_cov_cv, nis_cv, _, _ = kalman_filter(
    true_states_cv, x0, P0, F, Q, H, R
)

time_idx = np.arange(len(meas_cv))

# Trajectory
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(true_pos_cv[:, 0], true_pos_cv[:, 1],
        color='tab:blue', linewidth=2, label='True Position')
sc = ax.scatter(meas_cv[:, 0], meas_cv[:, 1],
                c=time_idx, cmap='viridis', s=80, alpha=0.9,
                marker='x', linewidths=1.5, label='Measurements')
ax.plot(est_cv[:, 0], est_cv[:, 1],
        color='tab:red', linewidth=2, label='KF Estimate')
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Time step')
ax.set_title('Constant Velocity — Trajectory')
ax.set_xlabel('X Position (m)')
ax.set_ylabel('Y Position (m)')
ax.grid(True)
ax.autoscale()
ax.legend()
plt.tight_layout()
plt.show()

rmse_cv = np.sqrt(np.mean(np.sum((est_cv - true_pos_cv) ** 2, axis=1)))
print(f'RMSE: {rmse_cv:.4f} m')
print(f'Mean NIS: {np.mean(nis_cv):.3f}  (expected ≈ {H.shape[0]})')

## Scenario 2: Circular Trajectory

The true trajectory follows uniform circular motion ($p_x = R\cos\omega t,\; p_y = R\sin\omega t$), but the filter uses a constant-velocity model. A CV model cannot represent sustained turning — the direction change is continuous and structural, not just a tuning problem. The NIS plot reveals that the filter's uncertainty bookkeeping is wrong, as innovations are far larger than the filter expects.

In [ ]:
# Circular trajectory parameters
# omega=0.2 rad/s → centripetal acc = 0.8 m/s², 8x sigma_acc — filter clearly can't keep up
# 25 steps covers ~0.8 of a circle, enough to see lag develop without looping back
num_steps_circ = 25
radius = 20.0
omega  = 0.2

# Initial state: position at t=0 on the circle, moving tangentially
x0_circ = np.array([radius, 0.0, 0.0, radius * omega])

true_states_circ = circular_true_position(num_steps_circ, dt, radius, omega)

np.random.seed(42)
true_pos_cb, meas_cb, est_cb, _, _, _, nis_cb, _, _ = kalman_filter(
    true_states_circ, x0_circ, P0, F, Q, H, R
)

rmse_circ = np.sqrt(np.mean(np.sum((est_cb - true_pos_cb) ** 2, axis=1)))
print(f'RMSE: {rmse_circ:.4f} m')
print(f'Mean NIS: {np.mean(nis_cb):.3f}  (expected ≈ {H.shape[0]})')

In [ ]:
time_idx_c = np.arange(num_steps_circ)

# 95% chi-squared confidence bounds (2 DOF — 2D measurements)
nis_dof = H.shape[0]
nis_lower = chi2.ppf(0.025, df=nis_dof)
nis_upper = chi2.ppf(0.975, df=nis_dof)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Left: Trajectory
ax = axes[0]
sc = ax.scatter(meas_cb[:, 0], meas_cb[:, 1],
                c=time_idx_c, cmap='viridis', s=80, alpha=0.9,
                marker='x', linewidths=1.5, label='Measurements', zorder=1)
ax.plot(true_pos_cb[:, 0], true_pos_cb[:, 1],
        color='tab:blue', linewidth=2, label='True Position', zorder=2)
ax.plot(est_cb[:, 0], est_cb[:, 1],
        color='tab:red', linewidth=2, label='KF Estimate', zorder=3)
ax.scatter(true_pos_cb[0, 0], true_pos_cb[0, 1],
           color='green', s=120, zorder=5, label='Start')
mid = len(true_pos_cb) // 4
ax.annotate('',
            xy=true_pos_cb[mid + 1],
            xytext=true_pos_cb[mid],
            arrowprops=dict(arrowstyle='->', color='tab:blue', lw=2))
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Time step')
ax.set_aspect('equal')
ax.set_title('Trajectory')
ax.set_xlabel('X Position (m)')
ax.set_ylabel('Y Position (m)')
ax.grid(True)
ax.legend()

# Right: NIS
ax = axes[1]
ax.plot(time_idx_c, nis_cb, color='tab:blue', linewidth=1.5, alpha=0.8, label='NIS')
ax.axhline(y=nis_dof,    color='black', linestyle='--', linewidth=1.5, label=f'Expected NIS = {nis_dof}')
ax.axhline(y=nis_upper,  color='tab:red',   linestyle=':', linewidth=1.5, label=f'95% upper bound ({nis_upper:.2f})')
ax.axhline(y=nis_lower,  color='tab:green', linestyle=':', linewidth=1.5, label=f'95% lower bound ({nis_lower:.2f})')
ax.set_title('Normalised Innovation Squared (NIS)')
ax.set_xlabel('Time step')
ax.set_ylabel('NIS')
ax.grid(True)
ax.legend()

plt.suptitle(f'Circular Trajectory — CV Kalman Filter (σ_acc = {sigma_acc})', fontsize=20)
plt.tight_layout()
plt.show()